# Digitra V2 — temiz eğitim hattı
V1'e dokunmaz. DINOv3 + aynı örneğin 422D landmark füzyonu ve ayrı J/Z/OTHER temporal modeli üretir. Test kullanıcıları eğitim, checkpoint, füzyon, kalibrasyon veya eşik seçimine girmez.

In [ ]:
import torch
assert torch.cuda.is_available(), 'GPU çalışma zamanı seçilmedi'
print(torch.cuda.get_device_name(0))
!pip -q install kagglehub mediapipe timm joblib scikit-learn onnx onnxruntime-gpu

## Gerekli dosyalar
Colab `/content` köküne `prepare_digitra_v2_static.py`, `train_digitra_v2_static.py`, `prepare_digitra_v2_dynamic.py`, `train_digitra_v2_dynamic.py`, `hand_landmarker.task` ve açılmış `DIGITRA_LANDMARK_V1/` klasörünü koy.

In [ ]:
import kagglehub
STATIC_ROOT = kagglehub.dataset_download('piotrpopis/asl-hands') + '/images'
DYNAMIC_ROOT = kagglehub.dataset_download('signnteam/asl-sign-language-alphabet-videos-j-z') + '/SigNN Video Data'
print(STATIC_ROOT, DYNAMIC_ROOT, sep='\n')

In [ ]:
!python /content/prepare_digitra_v2_static.py \
  --images-root "{STATIC_ROOT}" --task /content/hand_landmarker.task \
  --output /content/DIGITRA_V2_STATIC --workers 8

In [ ]:
# Signer-disjoint geliştirme koşusu: hiperparametre/epoch seçimi.
!python /content/train_digitra_v2_static.py \
  --manifest /content/DIGITRA_V2_STATIC/manifest.csv \
  --features /content/DIGITRA_V2_STATIC/paired_landmark_features_v2.npz \
  --v1-bundle /content/DIGITRA_LANDMARK_V1 \
  --output /content/DIGITRA_V2_STATIC_DEVELOPMENT \
  --image-size 320 --batch-size 96 --workers 8 \
  --head-epochs 2 --full-epochs 12 --skip-onnx

In [ ]:
# Development val ile seçilen epoch sayısını FULL_EPOCHS'e yaz.
# Signer 11 kalibrasyonda, signer 14/15 kilitli testte kalır.
FULL_EPOCHS = 7
!python /content/train_digitra_v2_static.py \
  --manifest /content/DIGITRA_V2_STATIC/manifest.csv \
  --features /content/DIGITRA_V2_STATIC/paired_landmark_features_v2.npz \
  --v1-bundle /content/DIGITRA_LANDMARK_V1 \
  --output /content/DIGITRA_V2_STATIC_FINAL \
  --image-size 320 --batch-size 96 --workers 8 \
  --head-epochs 2 --full-epochs {FULL_EPOCHS} \
  --include-development-val --calibration-signers 11 \
  --fixed-final-checkpoint --skip-onnx

In [ ]:
!python /content/prepare_digitra_v2_dynamic.py \
  --video-root "{DYNAMIC_ROOT}" --task /content/hand_landmarker.task \
  --output /content/DIGITRA_V2_DYNAMIC --workers 8
!python /content/train_digitra_v2_dynamic.py \
  --positive /content/DIGITRA_V2_DYNAMIC/dynamic_positive_v2.npz \
  --static-manifest /content/DIGITRA_V2_STATIC/manifest.csv \
  --static-features /content/DIGITRA_V2_STATIC/paired_landmark_features_v2.npz \
  --output /content/DIGITRA_V2_DYNAMIC_FINAL --epochs 90 --batch-size 96 --workers 4

In [ ]:
import json, pathlib
for path in [pathlib.Path('/content/DIGITRA_V2_STATIC_FINAL/metrics.json'), pathlib.Path('/content/DIGITRA_V2_DYNAMIC_FINAL/metrics.json')]:
    report = json.loads(path.read_text())
    print(path.parent.name, json.dumps(report['locked_test'], indent=2))